<a href="https://colab.research.google.com/github/Banty825413/Ml-and-DL-projects-/blob/main/fakenewsclassifierusinglstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_csv('/content/WELFake_Dataset.csv', on_bad_lines='skip', engine='python')

In [ ]:
df.shape


(4893, 4)

In [ ]:
df.loc[:, 'label'] = pd.to_numeric(df['label'], errors='coerce')
df=df.dropna()

In [ ]:
# Getting independent features
x=df.drop('label',axis=1)

In [ ]:
y=df['label']

In [ ]:
import tensorflow as tf

In [ ]:
from tensorflow.keras.layers import Embedding,LSTM,Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot

In [ ]:
vocab = 5000

In [ ]:
# ONe Hot Representaino
messsages=x.copy()

In [ ]:
messsages.reset_index(inplace=True)

In [ ]:

import nltk
import re
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Datapreprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus=[]
for i in range (0,len(messsages)):

  review = re.sub('[^a-zA-Z]',' ',messsages['title'][i])
  review=review.lower()
  review=review.split()

  review=[
      ps.stem(word)
      for word in review
      if not word in stopwords.words('english')] #Those word which are not in stopwords like 'the', 'that','is'etc
  review = ' '.join(review)  #AGain it become sentence,like: [i am boy]
  corpus.append(review)

In [ ]:
corpus[1]

'unbeliev obama attorney gener say charlott rioter peac protest home state north carolina video'

In [ ]:
# NOw one HOTEncoding
one_hot_rep=[one_hot(words,vocab) for words in corpus]
one_hot_rep[2] #Lets see how it changes

[4601, 4461, 3370, 1483, 4899, 4677, 2735, 3846, 3855, 2205, 3806, 330]

In [ ]:
# Padseqeuneces
max_len=20
embedded_doc=pad_sequences(one_hot_rep,padding='pre',maxlen=max_len)
print(embedded_doc[2]) #let see the example how it 0 is added infront of words if it has less than 20 words

[   0    0    0    0    0    0    0    0 4601 4461 3370 1483 4899 4677
 2735 3846 3855 2205 3806  330]


In [ ]:
# Creating the model
embedding_feature=40
model=Sequential()
model.add(Embedding(vocab,embedding_feature,input_length=max_len))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compiling the model
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
len(embedded_doc),y.shape

(4892, (4892,))

In [ ]:
import numpy as np
X_final=np.array(embedded_doc)
y_final=np.array(y, dtype=int) # Convert y to integer type
X_final.shape,y_final.shape  #PRINTING DIMENSION OF X AND Y DATA

((4892, 20), (4892,))

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_final,y_final,test_size=0.33,random_state=42)



In [ ]:
# Finally training the data
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.6591 - loss: 0.6231 - val_accuracy: 0.8130 - val_loss: 0.4625
Epoch 2/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8587 - loss: 0.3520 - val_accuracy: 0.8254 - val_loss: 0.3639
Epoch 3/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9152 - loss: 0.2183 - val_accuracy: 0.8594 - val_loss: 0.3211
Epoch 4/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9506 - loss: 0.1456 - val_accuracy: 0.8650 - val_loss: 0.3297
Epoch 5/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9643 - loss: 0.1052 - val_accuracy: 0.8539 - val_loss: 0.4026
Epoch 6/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9771 - loss: 0.0721 - val_accuracy: 0.8508 - val_loss: 0.3941
Epoch 7/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9838 - loss: 0.0576 - val_accuracy: 0.8520 - val_loss: 0.4398
Epoch 8/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9835 - loss: 0.0488 - val_accuracy: 0.8520 - val_l

In [ ]:
#Addding DropOut
from tensorflow.keras.layers import Dropout

In [ ]:
# Creating the model
embedding_feature=40
model=Sequential()
model.add(Embedding(vocab,embedding_feature,input_length=max_len))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Performance Matrix
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score
y_pred=model.predict(X_test)

confusion_matrix(y_test,y_pred.round())

51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


array([[671, 125],
       [111, 708]])

In [ ]:
print((671+708)/(708+125+671+111))

0.8538699690402477


In [ ]:
print(accuracy_score(y_pred.round(),y_test))

0.8538699690402477


In [ ]:
print(f1_score(y_pred.round(),y_test))

0.8571428571428571
